# MiniDance · 数据处理（Colab）

从 OpenVid-1M 筛选出约 4.4 万条风景片段，用冻结的 Wan2.2 VAE 编码成 latent，结果存到 Google 网盘。

- **请在 Colab 网页版中运行**（需要挂载 Google 网盘，VS Code 插件目前不支持）
- 运行时：**GPU → A100**（L4 / T4 也能跑，只是更慢）
- 网盘需要约 **10GB** 空闲空间
- 中途断开也没关系：重新运行全部单元格，会从断开处继续
- 原始视频不会落盘：下载一批、编码一批、删除一批

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 挂载网盘，获取代码

In [ ]:
REPO_URL = 'https://github.com/JuniverseLi/minidance.git'
DRIVE_DIR = '/content/drive/MyDrive/minidance'   # 所有产出都放在这里

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    # VS Code 的 Colab 插件目前不支持挂载网盘；不挂载的话结果只在临时磁盘上，断开即丢失
    raise RuntimeError('网盘挂载失败，请在 Colab 网页版中打开本 notebook 运行') from e
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists('/content/minidance'):
    !git clone -q {REPO_URL} /content/minidance
else:
    # 强制同步到 GitHub 最新版本（即使远程历史被改写也能更新）
    !git -C /content/minidance fetch -q origin
    !git -C /content/minidance reset -q --hard origin/main
%cd /content/minidance

In [ ]:
# Colab 自带 torch，只补装缺的包
!pip install -q pylance diffusers safetensors opencv-python-headless

In [ ]:
# HF token（可选，只需读权限）：不设置也能跑，但可能会被 HF 限速
# 优先读 Colab 左侧「🔑 Secrets」里的 HF_TOKEN，没有的话手动粘贴，直接回车则跳过
import os
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass('HF token（回车跳过）: ').strip()
if token:
    os.environ['HF_TOKEN'] = token
print('HF_TOKEN set' if token else '匿名下载')

## 2. 元数据：拉取并筛选（约 10 分钟，结果缓存到网盘，只需跑一次）

In [ ]:
META = f'{DRIVE_DIR}/scenery.parquet'
if not os.path.exists(META):
    !python dataset/fetch_openvid_meta.py --out /content/openvid_meta
    !python -c "import pandas as pd, glob; pd.concat(pd.read_parquet(f) for f in sorted(glob.glob('/content/openvid_meta/*.parquet'))).to_parquet('/content/openvid_meta_all.parquet')"
    !python dataset/filter_openvid.py --meta /content/openvid_meta_all.parquet --out {META}
else:
    print('已有', META)

## 3. 编码 latent

每 512 条打包成一个分片（约 110MB），共约 85 个分片，写到 `DRIVE_DIR/latents/`。
已经完成的分片会自动跳过。脚本会打印实时速度和剩余时间（ETA）。

In [ ]:
!python dataset/encode_latents.py     --parquet {META}     --out {DRIVE_DIR}/latents     --batch 16 --workers 8

In [ ]:
# 检查结果
import glob, pandas as pd
shards = sorted(glob.glob(f'{DRIVE_DIR}/latents/shard_*.safetensors'))
n = sum(len(pd.read_parquet(s.replace('.safetensors', '.parquet'))) for s in shards)
size = sum(os.path.getsize(s) for s in shards) / 2**30
print(f'{len(shards)} 个分片，{n:,} 条片段，{size:.1f} GB')